# E-commerce Data Analysis - Exploratory Data Analysis

This notebook demonstrates comprehensive EDA techniques using Python and pandas, featuring:
- Data generation and manipulation with pandas
- Data joining operations
- Three different types of visualizations
- Statistical analysis and insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib for better plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")

## 1. Data Generation

Let's create synthetic e-commerce data including customer and transaction information.

In [ ]:
def generate_sample_data():
    """Generate sample datasets for EDA demonstration"""
    
    # Generate customers data
    n_customers = 1000
    customers = pd.DataFrame({
        'customer_id': range(1, n_customers + 1),
        'name': [f'Customer_{i}' for i in range(1, n_customers + 1)],
        'age': np.random.randint(18, 80, n_customers),
        'gender': np.random.choice(['M', 'F', 'Other'], n_customers, p=[0.48, 0.48, 0.04]),
        'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 
                                'Philadelphia', 'San Antonio', 'San Diego'], n_customers),
        'registration_date': pd.date_range(
            start=datetime.now() - timedelta(days=365*2), 
            end=datetime.now(), 
            periods=n_customers
        ),
        'annual_income': np.random.lognormal(mean=10.5, sigma=0.5, size=n_customers).astype(int)
    })
    
    # Generate transactions data
    n_transactions = 5000
    transactions = pd.DataFrame({
        'transaction_id': range(1, n_transactions + 1),
        'customer_id': np.random.choice(customers['customer_id'], n_transactions),
        'transaction_date': pd.date_range(
            start=datetime.now() - timedelta(days=90), 
            end=datetime.now(), 
            periods=n_transactions
        ),
        'amount': np.random.lognormal(mean=3.5, sigma=1.2, size=n_transactions),
        'category': np.random.choice(['Electronics', 'Clothing', 'Food', 'Books', 'Home', 'Sports'], 
                                   n_transactions, p=[0.25, 0.20, 0.15, 0.15, 0.15, 0.10]),
        'payment_method': np.random.choice(['Credit Card', 'Debit Card', 'PayPal', 'Cash'], 
                                         n_transactions, p=[0.4, 0.3, 0.2, 0.1])
    })
    
    return customers, transactions

# Generate the data
customers, transactions = generate_sample_data()

print(f"Generated {len(customers):,} customers and {len(transactions):,} transactions")
print("\nCustomers data preview:")
print(customers.head())
print("\nTransactions data preview:")
print(transactions.head())

## 2. Data Joining

Let's join the customers and transactions data to enable comprehensive analysis.

In [ ]:
# Join customers and transactions data
merged_data = transactions.merge(customers, on='customer_id', how='left')

print(f"Merged data shape: {merged_data.shape}")
print("\nMerged data info:")
print(merged_data.info())
print("\nSample of merged data:")
print(merged_data.head())

## 3. Exploratory Data Analysis with 3 Visualizations

Now let's create three different types of visualizations to explore our data.

### Plot 1: Distribution of Transaction Amounts

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=merged_data, x='amount', bins=50, alpha=0.7)
plt.axvline(merged_data['amount'].median(), color='red', linestyle='--', 
            label=f'Median: ${merged_data["amount"].median():.2f}')
plt.axvline(merged_data['amount'].mean(), color='orange', linestyle='--', 
            label=f'Mean: ${merged_data["amount"].mean():.2f}')
plt.title('Distribution of Transaction Amounts', fontsize=14, fontweight='bold')
plt.xlabel('Transaction Amount ($)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Plot 2: Transaction Categories by Age Groups

In [ ]:
# Create age groups
merged_data['age_group'] = pd.cut(merged_data['age'], 
                                bins=[0, 25, 35, 50, 65, 100], 
                                labels=['18-25', '26-35', '36-50', '51-65', '65+'])

# Create the plot
plt.figure(figsize=(12, 8))
category_age = merged_data.groupby(['category', 'age_group']).size().unstack(fill_value=0)
category_age.plot(kind='bar', stacked=True, figsize=(12, 8))
plt.title('Transaction Categories by Age Group', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Number of Transactions')
plt.legend(title='Age Group', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Plot 3: Monthly Transaction Trends

In [ ]:
# Prepare monthly data
merged_data['month'] = merged_data['transaction_date'].dt.to_period('M')
monthly_sales = merged_data.groupby('month')['amount'].agg(['sum', 'count'])
monthly_sales.index = monthly_sales.index.to_timestamp()

# Create dual-axis plot
fig, ax1 = plt.subplots(figsize=(12, 8))
ax2 = ax1.twinx()

line1 = ax1.plot(monthly_sales.index, monthly_sales['sum'], 'b-o', linewidth=2, label='Total Sales ($)')
line2 = ax2.plot(monthly_sales.index, monthly_sales['count'], 'r-s', linewidth=2, label='Transaction Count')

ax1.set_title('Monthly Transaction Trends', fontsize=14, fontweight='bold')
ax1.set_xlabel('Month')
ax1.set_ylabel('Total Sales ($)', color='b')
ax2.set_ylabel('Transaction Count', color='r')

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Statistical Analysis and Insights

In [ ]:
print("="*60)
print("COMPREHENSIVE DATA ANALYSIS REPORT")
print("="*60)

# Basic statistics
print(f"\n📊 Dataset Overview:")
print(f"   • Total customers: {customers['customer_id'].nunique():,}")
print(f"   • Total transactions: {len(transactions):,}")
print(f"   • Date range: {merged_data['transaction_date'].min().strftime('%Y-%m-%d')} to {merged_data['transaction_date'].max().strftime('%Y-%m-%d')}")
print(f"   • Total revenue: ${merged_data['amount'].sum():,.2f}")

# Customer demographics
print(f"\n👥 Customer Demographics:")
print(f"   • Average age: {merged_data['age'].mean():.1f} years")
print(f"   • Age range: {merged_data['age'].min()}-{merged_data['age'].max()} years")
print(f"   • Gender distribution:")
for gender, count in merged_data.drop_duplicates('customer_id')['gender'].value_counts().items():
    print(f"     - {gender}: {count:,} ({count/merged_data['customer_id'].nunique()*100:.1f}%)")

# Transaction analysis
print(f"\n💰 Transaction Analysis:")
print(f"   • Average transaction: ${merged_data['amount'].mean():.2f}")
print(f"   • Median transaction: ${merged_data['amount'].median():.2f}")
print(f"   • Transaction std dev: ${merged_data['amount'].std():.2f}")
print(f"   • Highest transaction: ${merged_data['amount'].max():.2f}")
print(f"   • Lowest transaction: ${merged_data['amount'].min():.2f}")

# Category analysis
print(f"\n🛍️ Category Performance:")
category_stats = merged_data.groupby('category')['amount'].agg(['count', 'sum', 'mean']).sort_values('sum', ascending=False)
for category in category_stats.index:
    stats = category_stats.loc[category]
    print(f"   • {category}: {stats['count']} transactions, ${stats['sum']:.2f} total, ${stats['mean']:.2f} avg")

# Payment method analysis
print(f"\n💳 Payment Methods:")
payment_stats = merged_data.groupby('payment_method')['amount'].agg(['count', 'sum']).sort_values('sum', ascending=False)
for method in payment_stats.index:
    stats = payment_stats.loc[method]
    print(f"   • {method}: {stats['count']} transactions ({stats['count']/len(merged_data)*100:.1f}%), ${stats['sum']:.2f} total")

## 5. Save Results

In [ ]:
# Save datasets
customers.to_csv('customers.csv', index=False)
transactions.to_csv('transactions.csv', index=False)
merged_data.to_csv('merged_data.csv', index=False)

print("✅ Data saved successfully:")
print("   • customers.csv")
print("   • transactions.csv") 
print("   • merged_data.csv")

# Display final summary
summary = pd.DataFrame({
    'Dataset': ['Customers', 'Transactions', 'Merged Data'],
    'Records': [len(customers), len(transactions), len(merged_data)],
    'Columns': [len(customers.columns), len(transactions.columns), len(merged_data.columns)]
})
print("\n📋 Final Summary:")
print(summary)